#### Relevant Imports

# Notebook 6 — Loading CSV Data into a SQL Database

## What You Will Learn

Data pipelines need a way to get external files into a queryable database. This notebook shows you how to ingest a CSV file into SQLite with **correct data types** — a foundational step before any SQL-based RAG can run.

### Topics Covered

| Step | Tool | Purpose |
|---|---|---|
| **Load CSV** | `pandas.read_csv()` | Read file and auto-detect column types |
| **Inspect dtypes** | `DataFrame.dtypes` | Understand what pandas inferred |
| **Map to SQL types** | Custom `map_dtype()` function | Convert pandas types → SQLAlchemy types |
| **Write to DB** | `DataFrame.to_sql()` | Persist data with explicit type definitions |
| **Verify** | SQL SELECT query | Confirm the data is correct in the database |

### Type Mapping Reference

| Pandas dtype | SQLAlchemy Type |
|---|---|
| `int64` | `Integer` |
| `float64` | `Float` |
| `bool` | `Boolean` |
| `datetime64` | `DateTime` |
| `object` (string) | `String` |

### Skills You Will Build

- Read and inspect CSV files using `pandas`
- Write a dtype-to-SQLAlchemy type mapper
- Ingest a DataFrame into SQLite with `to_sql()` and proper type enforcement
- Verify persistence by querying the newly created table

> **Why it matters:** Correct type mapping prevents silent errors (e.g., numbers stored as strings) that would break downstream SQL queries and LLM-generated aggregations.

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.types import (
    Integer, Float, String, DateTime, Boolean
)
import pandas as pd
import os


In [ ]:
# Map pandas dtypes to SQLAlchemy types
def map_dtype(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return Integer()
    elif pd.api.types.is_float_dtype(dtype):
        return Float()
    elif pd.api.types.is_bool_dtype(dtype):
        return Boolean()
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return DateTime()
    else:
        return String()  # Default for string

>Load the CSV file into a pandas data frame  
>data frame reads with the relevant types. If needed it can be adjusted (number, date etc)  
>mapping can be done to SQLAlchemy

In [ ]:
# Load CSV into DataFrame
csv_file = 'Student_Personal_Details.csv'
Data = pd.read_csv(csv_file)
print (Data.dtypes)

In [ ]:
# Create SQLite engine using SQLAlchemy
DB_File = "Sample_2 copy.db"

if os.path.exists (DB_File):
    sql_engine = create_engine("sqlite:///"+DB_File)
    # conn_1 = sql_engine.connect ()
else:
    print ("DB Files does not exist")

>mapping of types

In [ ]:
# Build SQLAlchemy type mapping
sql_types = {col: map_dtype(dtype) for col, dtype in Data.dtypes.items()}
print (sql_types)

>Use pandas method to ingest the data into DB  
>Since it is done via sqlalchemy, it can be used for any DB that is linked

In [ ]:
# Write to table using  SQLAlchemy types
table_name = 'Student_Details'
Data.to_sql(table_name, con=sql_engine, if_exists='replace', index=False, dtype=sql_types)

>Check if the data is persisting in the database

In [ ]:
# Launch a query on newly created table
conn = sql_engine.connect ()

Result = conn.execute (text("select * from Student_Details limit 10"))

for row in Result:
    print (row)